## Retrieval-Augmented Generation (RAG)

Large Language Models (LLMs) generate responses primarily from knowledge acquired during training. Consequently, they do not automatically have access to specialised or private collections, such as a locally stored research corpus. **Retrieval-Augmented Generation (RAG)** addresses this limitation by combining an LLM with an external information retrieval system.[^1]

Instead of relying exclusively on the model's internal knowledge, a RAG system first **retrieves information relevant to the user's query** and then provides this information to the LLM as additional context. This is particularly useful when working with specialised corpora that were not part of the model's training data, or collections that are too large to fit into the model's context window.[^1]

### A simplified RAG pipeline can be represented as:

> **User question → retrieve relevant documents → add documents to the context → LLM → generated answer**

RAG does not normally retrain the language model on the external collection. Instead, the external data are made available to the model **at inference time**.[^1]





### Indexing the corpus

Before documents can be retrieved, the corpus must be prepared for efficient search. In a typical embedding-based RAG workflow, this consists of four main operations described by LangChain as **load, split, embed, and store**:[^1]

1. **Load** – import the source documents.
2. **Split** – divide long documents into smaller units or chunks where necessary.
3. **Embed** – convert each document into a numerical vector representing its semantic content.
4. **Store** – save the embeddings and associated documents in a vector store such as Chroma.

For a corpus in which each CSV row already represents a short document, additional splitting may not be necessary. Each row can instead become a single retrievable document.

The resulting workflow is approximately:

**CSV → Documents → Embedding model → Vector store**

At query time, the process is reversed into a retrieval pipeline:

**Question → Query embedding → Similarity search → Relevant documents → LLM → Answer**


### Embedding-based RAG

The most common RAG architecture uses **vector embeddings** for retrieval. An embedding model transforms a piece of text into a numerical vector. Texts that are semantically similar should occupy relatively similar positions in this vector space.

For example, a document discussing *negative representations of minority groups* might be retrieved for a query about *hostile portrayals of minorities*, even when the exact vocabulary differs.

The basic retrieval process is:

```text
document → embedding ─┐
document → embedding ─┤
document → embedding ─┤
                      ├→ similarity → most relevant documents
question → embedding ─┘
```

Vector databases such as Chroma store these representations and provide efficient similarity search.[^2]

Embedding-based retrieval is particularly effective when the principal goal is to answer questions such as:

> *Which documents are semantically relevant to this question?*

It is relatively simple to implement, works well with unstructured text, and can be combined with metadata filtering. For example, semantic retrieval can be restricted to documents from a particular **year, newspaper, topic, category, or sentiment class**.

Its principal limitation is that **semantic similarity is not the same as an explicit relationship**. Two documents may be related through a person, organisation, event, temporal sequence, or other relationship even when their textual embeddings are not particularly similar.



# Example:  What does "self-reflection" mean in the context of agentic AI?

## Task decomoposition:

<b> Goal → reason about what needs to be done → create subtasks → execute them → inspect results → revise the plan → continue </b>

### Installations (skip if not neccessary)

In [1]:
!pip install -U \
    langchain \
    langchain-openai \
    langchain-chroma \
    langchain-docling \
    langchain-community \
    langchain-text-splitters \
    chromadb \
    docling \
    beautifulsoup4

INFO: pip is looking at multiple versions of docling-slim[convert-core,feat-chunking,service-client] to determine which version is compatible with other requirements. This could take a while.
  Using cached numpy-2.2.6-cp312-cp312-macosx_14_0_arm64.whl.metadata (62 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 732.9/732.9 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 567.0/567.0 kB 30.6 MB/s eta 0:00:00
Using cached numpy-2.2.6-cp312-cp312-macosx_14_0_arm64.whl (5.1 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.5.4
    Uninstalling langchain-core-1.5.4:
      Successfully uninstalled langchain-core-1.5.4
  Attempting uninstall: langchain-openai
    Found existing installation: langchain-openai 1.5.1
    Uninstalling langchain-openai-1.5.1:
      Successfully uninstal

In [2]:
import sys
!{sys.executable} -m pip install -U langchain-community

In [5]:
import sys
import numpy as np

print(sys.executable)
print(np.__version__)
print(np.__file__)

/opt/anaconda3/bin/python
2.2.6
/opt/anaconda3/lib/python3.12/site-packages/numpy/__init__.py


In [7]:
!pip install numpy==1.26.4

  Using cached numpy-1.26.4-cp312-cp312-macosx_11_0_arm64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-macosx_11_0_arm64.whl (13.7 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
streamlit 1.37.1 requires protobuf<6,>=3.20, but you have protobuf 7.35.1 which is incompatible.


### Main imports

In [10]:
%pip install --force-reinstall "numpy<2" "pandas>=2.2,<3"


  Using cached numpy-1.26.4-cp312-cp312-macosx_11_0_arm64.whl.metadata (61 kB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
  Using cached six-1.17.0-py2.py3-none-any.whl.metadata (1.7 kB)
Using cached numpy-1.26.4-cp312-cp312-macosx_11_0_arm64.whl (13.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 30.4 MB/s eta 0:00:00 0:00:01
Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl (229 kB)
Using cached six-1.17.0-py2.py3-none-any.whl (11 kB)
  Attempting uninstall: pytz
    Found existing installation: pytz 2024.1
    Uninstalling pytz-2024.1:
      Successfully uninstalled pytz-2024.1
  Attempting uninstall: tzdata
    Found existing installation: tzdata 2023.3
    Uninstalling tzdata-2023.3:
      Successfully uninstalled tzdata-2023.3
  Attempting uninstall: six
    Found existing installation: six 1.16.0
    Uninstalling six-1.16.0:
      Successfully uninstalled six-1.16.0
  Attempting uninstall: numpy
    Found existing

In [2]:
import os
import warnings
import logging
from langchain_openai import ChatOpenAI, OpenAIEmbeddings 
from langchain_chroma import Chroma
import bs4
from langchain.agents import AgentState, create_agent
from langchain_docling import DoclingLoader
from langchain.messages import MessageLikeRepresentation
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.tools import tool

### Environment setup

For this step you will need to: 
- get a Langchain API Key (https://docs.langchain.com/oss/python/deepagents/rag)
- be added DHInfra project by Florian and get DHInfa API kez

In [8]:
import os
from openai import OpenAI

In [5]:
os.environ["LANGCHAIN_API_KEY"] = "" # insert your own Langchain key
os.environ["LANGCHAIN_TRACING_V2"] = "false"  # <-- FIX 1: Disabled to prevent 403 error
os.environ["LANGCHAIN_PROJECT"] = "DHInfra-Tracing-Demo"
os.environ["LANGSMITH_DISABLE_RUN_COMPRESSION"] = "true"
os.environ["USER_AGENT"] = "my_agent"
os.environ["DHINFRA_API_KEY"] = "" # insert DHInfra key

In [6]:
# <-- FIX 2: Custom class to prevent the 422 "null content" error
class SanitizedChatOpenAI(ChatOpenAI):
    def _get_request_payload(self, input_, *args, **kwargs):
        payload = super()._get_request_payload(input_, *args, **kwargs)
        if "messages" in payload:
            for msg in payload["messages"]:
                if msg.get("content") is None:
                    msg["content"] = ""
        return payload

# Initialize chat model using the sanitized class
model = SanitizedChatOpenAI(
    model="qwen3.5-397b",
    openai_api_key=os.environ["DHINFRA_API_KEY"],
    openai_api_base="https://api.dhinfra.uni-graz.at/v1",
    model_kwargs={"parallel_tool_calls": False}
)

# Initialize embedding model
embeddings = OpenAIEmbeddings(
    model="qwen3-embedding-8b",
    openai_api_key=os.environ["DHINFRA_API_KEY"],
    openai_api_base="https://api.dhinfra.uni-graz.at/v1"
)

# Initialize Chroma vector store
vector_store = Chroma(
    collection_name="rag_collection",
    embedding_function=embeddings
)

print("Chat model (Qwen), embedding model (Qwen3-Embedding-8B), and Chroma vector store setup done")

Chat model (Qwen), embedding model (Qwen3-Embedding-8B), and Chroma vector store setup done


### this show us aann LLM response (without a specialized RAG) to the question "What is self-reflection?"

In [9]:
# Initialize client pointing to the DH-Infra API
client = OpenAI(
    api_key=os.environ.get("DHINFRA_API_KEY"),
    base_url="https://api.dhinfra.uni-graz.at/v1"
)

# Call the chat completion endpoint
response = client.chat.completions.create(
    model="qwen3.5-397b",
    messages=[
        {"role": "user", "content": "What is Self-Reflection?"}
    ]
)

# Print response content
print(response.choices[0].message.content)



**Self-reflection** is the conscious process of examining your own thoughts, feelings, actions, and motivations. It is the ability to "step back" and observe yourself as if you were a third party, allowing you to understand *why* you do what you do and *how* you can improve.

The ancient Greek philosopher Socrates famously said, **"The unexamined life is not worth living."** Self-reflection is the tool used to examine that life.

Here is a breakdown of what self-reflection entails, why it matters, and how to practice it effectively.

---

### 1. The Core Components
Self-reflection is not just daydreaming or worrying; it is a structured mental activity. It usually involves three stages:
*   **Awareness:** Noticing what happened (e.g., "I got angry during that meeting").
*   **Analysis:** Understanding why it happened (e.g., "I felt threatened because my expertise was questioned").
*   **Adjustment:** Deciding what to do differently next time (e.g., "Next time, I will take a deep breat

### Adding a custom RAG (in this case from a website)

In [ ]:
import warnings
import logging
import os

# 1. Suppress general warnings FIRST (fixes the langchain-community warning)
warnings.filterwarnings("ignore")
# Optional: suppresses the Hugging Face token warningos.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1" 

# NOW we can import the rest safely
from langchain_community.vectorstores.utils import filter_complex_metadata
from transformers import logging as tf_logging

# 2. Suppress Docling logs & HF Transformers logs
logging.getLogger("docling").setLevel(logging.ERROR)
tf_logging.set_verbosity_error()

# Setup RAG: Load document using Docling
loader = DoclingLoader(file_path="https://lilianweng.github.io/posts/2023-06-23-agent/")
docs = loader.load()

# Split document into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
all_splits = text_splitter.split_documents(docs)

# Filter out complex nested metadata (dicts, lists) for ChromaDB compatibility
filtered_splits = filter_complex_metadata(all_splits)

# Index cleaned chunks in Chroma
_ = vector_store.add_documents(documents=filtered_splits)

print("Document loaded, split, and indexed successfully without warnings!")

In [ ]:
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

# 1. Redefine the retrieval tool returning a plain string
@tool
def retrieve_context(query: str) -> str:
    """Retrieve information to help answer a query."""
    retrieved_docs = vector_store.similarity_search(query, k=2)
    
    serialized = "\n\n".join(
        f"Source: {doc.metadata.get('source', 'Blog Post')}\nContent: {doc.page_content}"
        for doc in retrieved_docs
    )
    return serialized if serialized else "No relevant context found."

# 2. Create agent
tools = [retrieve_context]
system_prompt = (
    "You have access to a tool that retrieves context from a blog post. "
    "Use the tool to help answer user queries."
)

# Wir nutzen das sichere 'model', das du bereits in der ersten Zelle geladen hast!
agent = create_react_agent(
    model, 
    tools, 
    prompt=system_prompt
)

print("Agent created successfully!")

### What is self-reflection, in The context of agentic LLMs (answer with addes RAG)

In [ ]:
query = "What is Self-Reflection?"

for step in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

## Adapting the code for our own data

If it is a dataframe:

In [ ]:
import warnings
import logging
import os
import pandas as pd

warnings.filterwarnings("ignore")
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores.utils import filter_complex_metadata


In [ ]:
import pandas as pd
from langchain_core.documents import Document

### Load local CSV

In [ ]:
df = pd.read_csv("MigraAnno.csv")
df = df.drop(columns=["Unnamed: 0"], errors="ignore")

In [ ]:
print(df.columns)


### Convert CSV rows into LangChain Documents

In [ ]:
def clean_value(value):
    return "" if pd.isna(value) else value


docs = []

for _, row in df.iterrows():

    # Text used for semantic retrieval
    page_content = f"""
Topic: {clean_value(row['Topic'])}
Category: {clean_value(row['Category'])}

{clean_value(row['text'])}[''
""".strip()

    # Structured information stored alongside each document
    metadata = {
        "id": clean_value(row["id"]),
        "topic": clean_value(row["Topic"]),
        "name_original": clean_value(row["Name-original"]),
        "newspaper_title": clean_value(row["newspaper_title"]),
        "date": clean_value(row["date"]),
        "preceding_document": clean_value(row["preceding_document"]),
        "following_document": clean_value(row["following_document"]),
        "relevancy_proba": clean_value(row["Relevancy_proba"]),
        "sentiment": clean_value(row["sentiment"]),
        "year": clean_value(row["year"]),
        "category": clean_value(row["Category"]),
    }

    docs.append(
        Document(
            page_content=page_content,
            metadata=metadata
        )
    )

In [ ]:
print("Filtering metadata...")
filtered_docs = filter_complex_metadata(docs)
print(f"Filtering done: {len(filtered_docs)} documents")

print("Starting vector store indexing...")
batch_size = 100

for start in range(0, len(filtered_docs), batch_size):
    end = min(start + batch_size, len(filtered_docs))

    vector_store.add_documents(
        documents=filtered_docs[start:end]
    )

    print(f"Indexed {end}/{len(filtered_docs)} documents")

In [ ]:
results = vector_store.similarity_search(
    "negative attitudes towards minority groups",
    k=10
)

### If it is an XML (just a skeleton code)

More information:

LangChain provides an `UnstructuredXMLLoader` for converting XML files into LangChain `Document` objects. [[1]](#ref1)

For richly structured XML, particularly **TEI XML**, it can be advantageous to exploit the encoded document structure rather than treating the XML as undifferentiated text. Castellon, Chiffoleau, and Miasnikova demonstrate how XML-TEI digital editions can serve as the knowledge source for a RAG system. [[2]](#ref2)


### References

<a id="ref1"></a>
**[1]** LangChain. *UnstructuredXMLLoader*. `langchain_community.document_loaders`.  
https://reference.langchain.com/python/langchain-community/document_loaders/xml/UnstructuredXMLLoader

<a id="ref2"></a>
**[2]** Castellon, C., Chiffoleau, F., & Miasnikova, A. (2026). *Faire du neuf avec du balisé : Quand une édition TEI devient la mémoire d'un RAG*. *Anthology of Computers and the Humanities*, 4.  
https://anthology.ach.org/volumes/vol0004/faire-du-neuf-avec-du-balis-quand-une-dition-tei-devient-la/

In [ ]:
from pathlib import Path
from lxml import etree

from langchain_core.documents import Document
from langchain_community.vectorstores.utils import filter_complex_metadata


In [ ]:
TEI_NS = {"tei": "http://www.tei-c.org/ns/1.0"}

xml_folder = Path("data/letters")

docs = []


In [ ]:
for xml_file in xml_folder.glob("*.xml"):

    tree = etree.parse(str(xml_file))
    root = tree.getroot()

   
    # Extract correspondence metadata
 

    letter_id = root.get(
        "{http://www.w3.org/XML/1998/namespace}id",
        xml_file.stem
    )

    sender = root.xpath(
        "string(.//tei:correspAction[@type='sent']/tei:persName)",
        namespaces=TEI_NS
    ).strip()

    recipient = root.xpath(
        "string(.//tei:correspAction[@type='received']/tei:persName)",
        namespaces=TEI_NS
    ).strip()

    date = root.xpath(
        "string(.//tei:correspAction[@type='sent']/tei:date/@when)",
        namespaces=TEI_NS
    ).strip()

    sender_place = root.xpath(
        "string(.//tei:correspAction[@type='sent']/tei:placeName)",
        namespaces=TEI_NS
    ).strip()

    recipient_place = root.xpath(
        "string(.//tei:correspAction[@type='received']/tei:placeName)",
        namespaces=TEI_NS
    ).strip()

    # Extract letter text

    body_nodes = root.xpath(
        ".//tei:text/tei:body",
        namespaces=TEI_NS
    )

    if body_nodes:
        body_text = " ".join(
            " ".join(body_nodes[0].itertext()).split()
        )
    else:
        body_text = ""

   
    # Construct text for embeddings
   
    page_content = f"""
Sender: {sender}
Recipient: {recipient}
Date: {date}

{body_text}
""".strip()

 
    # Metadata
    
    metadata = {
        "id": letter_id,
        "sender": sender,
        "recipient": recipient,
        "date": date,
        "sender_place": sender_place,
        "recipient_place": recipient_place,
        "source_file": xml_file.name,
    }

    docs.append(
        Document(
            page_content=page_content,
            metadata=metadata
        )
    )

print(f"{len(docs)} letters loaded.")

In [ ]:
print(docs[0].page_content)

print("\nMETADATA:")
print(docs[0].metadata)

In [ ]:
filtered_docs = filter_complex_metadata(docs)

print(f"Indexing {len(filtered_docs)} letters...")

vector_store.add_documents(
    documents=filtered_docs
)

print("Indexing complete.")

### Test semantic retreival

In [ ]:
query = "What do the correspondents say about linguistics?"

results = vector_store.similarity_search(
    query,
    k=5
)

for i, doc in enumerate(results, start=1):
    
    print(f"\n--- Result {i} ---")
    print("Sender:", doc.metadata["sender"])
    print("Recipient:", doc.metadata["recipient"])
    print("Date:", doc.metadata["date"])
    print("Source:", doc.metadata["source_file"])
    
    print()
    print(doc.page_content[:500])

## We now worked on an embedding-based RAG. What  other types of RAGs exist? 

### Graph-based RAG

**Graph-based RAG (Graph RAG)** approaches retrieval from a different perspective. Instead of representing the collection primarily as independent vectors, information is represented through **entities (nodes) and relationships (edges)**.[^3]

For example, a newspaper corpus could contain relationships such as:

```text
Newspaper
    │
 published
    ▼
 Document ── has_topic ──→ Topic
    │
    ├──── mentions ──────→ Person
    │
    ├── has_sentiment ───→ Negative
    │
    └──── belongs_to ────→ Category
```

Retrieval can therefore follow relationships between entities rather than relying exclusively on semantic similarity.

This is particularly useful for questions involving several interconnected properties, for example:

> *Which topics were associated with negatively represented people in a particular newspaper during the 1930s?*

Such a question may require traversal through several relationships:

```text
Newspaper
    ↓
Documents
    ↓
People
    ↓
Sentiment
    ↓
Topics
```

Graph-based retrieval is therefore particularly useful when the research question concerns **how entities are connected**, rather than simply which documents are semantically similar.

### Embedding-based RAG vs. Graph RAG

The fundamental difference is therefore the type of information used to determine relevance:

|                          | Embedding-based RAG      | Graph-based RAG                        |
| ------------------------ | ------------------------ | -------------------------------------- |
| Representation           | Numerical vectors        | Nodes and relationships                |
| Retrieval                | Semantic similarity      | Graph relationships/traversal          |
| Typical question         | *Which texts discuss X?* | *How is X related to Y?*               |
| Unstructured text        | Very suitable            | Usually requires additional modelling  |
| Explicit relationships   | Weak                     | Strong                                 |
| Metadata filtering       | Yes                      | Relationships can be modelled directly |
| Complexity               | Relatively low           | Higher                                 |
| Multi-step relationships | Limited                  | Particularly suitable                  |

These approaches are not mutually exclusive. **Hybrid RAG architectures can combine embedding similarity with graph traversal.** For example, LangChain's graph retriever can begin with vector similarity and subsequently traverse relationships defined through document metadata.[^3]

Conceptually:

```text
Question → Embedding search → Relevant documents → Graph relationships → Related documents → LLM|
```

For some corpora, **embedding-based RAG can be a good option**. 

In the case of MigraAnno, The document `text` (newspaper article) can be embedded for semantic retrieval, while fields such as `Topic`, `newspaper_title`, `date`, `sentiment`, `year`, and `Category` can be retained as structured metadata.

The `preceding_document` and `following_document` fields additionally encode explicit relationships between documents. They could therefore later provide a natural basis for graph-based or hybrid retrieval.

In short:

> **Embedding-based RAG retrieves information because it is semantically similar; Graph RAG retrieves information because it is explicitly connected.**

A hybrid system can exploit both forms of evidence.



### References

<a id="ref1"></a>
**[1]** LangChain. *Retrieval-Augmented Generation (RAG) with Deep Agents*. LangChain Documentation.  
https://docs.langchain.com/oss/python/deepagents/rag

<a id="ref2"></a>
**[2]** Chroma. *Chroma Documentation*.  
https://docs.trychroma.com/

<a id="ref3"></a>
**[3]** LangChain. *Graph RAG / GraphRetriever*. LangChain Documentation.  
https://docs.langchain.com/oss/python/integrations/retrievers/graph_rag